# Geospatial Boundary Correction - Exploratory Data Analysis

This notebook performs the initial dataset exploration for the BhuMe Boundary Correction Take-Home challenge, focusing on the village **Vadnerbhairav** in Nashik.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import geopandas as gpd
import rasterio
from src.io import load, open_imagery

## 1. Load Village and Check General Info

In [ ]:
village_dir = Path("../data/34855_vadnerbhairav_chandavad_nashik")
village = load(village_dir)
print(f"Village Slug: {village.slug}")
print(f"Plots count: {len(village.plots)}")
print(f"Plots CRS: {village.plots.crs}")

## 2. Imagery and Boundary Raster Metadata

In [ ]:
with open_imagery(village.imagery_path) as img:
    print("--- Imagery Metadata ---")
    print(f"CRS: {img.crs}")
    print(f"Dimensions: {img.width}x{img.height}")
    print(f"Pixel Resolution: {img.transform[0]} m/px")

if village.boundaries_path:
    with open_imagery(village.boundaries_path) as bnd:
        print("\n--- Boundary Raster Metadata ---")
        print(f"CRS: {bnd.crs}")
        print(f"Dimensions: {bnd.width}x{bnd.height}")
        print(f"Pixel Resolution: {bnd.transform[0]} m/px")

## 3. Area Distributions & Ratios

We compare the drawn map area of the cadastre outlines with the recorded cultivable area (and total recorded area including pot-kharaba).

In [ ]:
# Compute drawn areas in metric CRS (EPSG:3857)
plots_3857 = village.plots.to_crs("EPSG:3857")
drawn_areas = plots_3857.geometry.area
print(f"Drawn Area (sqm) Median: {drawn_areas.median():.2f}")

recorded = village.plots['recorded_area_sqm']
print(f"Recorded Area (sqm) Median: {recorded.median():.2f}")

# Compute total recorded area including pot-kharaba uncultivable land
pot_kharaba_sqm = village.plots['pot_kharaba_ha'].fillna(0) * 10000
total_recorded_sqm = recorded.fillna(0) + pot_kharaba_sqm

# Area ratios
total_recorded_sqm = total_recorded_sqm.replace(0, np.nan)
ratios = drawn_areas / total_recorded_sqm
print(f"Median Area Ratio (Drawn / Total Recorded): {ratios.median():.4f}")
print(f"Fraction of plots with area ratios within [0.8, 1.2]: {((ratios >= 0.8) & (ratios <= 1.2)).mean() * 100:.2f}%")

## 4. Analyze Example Truth Offsets

In [ ]:
if village.example_truths is not None:
    lon = village.example_truths.geometry.iloc[0].centroid.x
    utm_zone = f'EPSG:{32600 + int((lon + 180) // 6) + 1}'
    
    plots_utm = village.plots.to_crs(utm_zone)
    truths_utm = village.example_truths.to_crs(utm_zone)
    
    dxs = []
    dys = []
    ious = []
    
    for pn in village.example_truths.index:
        if pn in plots_utm.index:
            t = truths_utm.loc[pn, 'geometry']
            o = plots_utm.loc[pn, 'geometry']
            
            inter = t.intersection(o).area
            union = t.union(o).area
            iou = inter / union if union > 0 else 0
            ious.append(iou)
            
            dx = t.centroid.x - o.centroid.x
            dy = t.centroid.y - o.centroid.y
            dxs.append(dx)
            dys.append(dy)
            
            print(f"Plot {pn}: IoU={iou:.4f}, dx={dx:.2f} m, dy={dy:.2f} m")
            
    print(f"\nMedian IoU: {np.median(ious):.4f}")
    print(f"Median global dx: {np.median(dxs):.2f} m")
    print(f"Median global dy: {np.median(dys):.2f} m")